# Теорія функціонала густини (DFT): практикум у PySCF

**Тема:** метод DFT (Кона–Шема), функціонали обміну–кореляції, їх сильні та слабкі сторони.

## Як влаштовано цей практикум

Робота йде *сходинками*: кожна наступна задача спирається на вміння, які ви отримали в попередній.

| Рівень | Що ви робите | Частини |
|---|---|---|
| 1. Запустити й прочитати | виконуєте готовий розрахунок, розумієте кожен рядок і кожне число | 1 |
| 2. Подивитись всередину | «розкладаєте» енергію та густину на складові | 2, 3 |
| 3. Порівняти | змінюєте функціонал / базис / сітку й бачите, що змінилось | 4, 9 |
| 4. Пояснити | пов'язуєте числа з фізикою (самодія, делокалізація, асимптотика потенціалу) | 5, 6, 7 |
| 5. Застосувати | рахуєте геометрію та енергії атомізації, порівнюєте з експериментом | 8 |
| 6. Оцінити | самостійно обираєте функціонал і обґрунтовуєте вибір | 10 |

**Правила виконання**

* У кожній частині є *теоретична довідка*, *приклад* (готовий код, його треба запустити й прочитати) та *завдання*.
* У клітинках із завданнями шукайте позначки `# TODO` — саме там треба дописати код.
* Перш ніж запускати код, **спробуйте передбачити результат** (хоча б знак і порядок величини). Це найкращий спосіб навчитися.
* Усі енергії — в атомних одиницях (**Hartree**, Ha), довжини в коді — в Å, якщо не сказано інше.
  Корисно: $1\ \text{Ha} = 27.2114\ \text{еВ} = 627.51\ \text{ккал/моль}$.
* Текстові відповіді пишіть у клітинках «**Ваша відповідь**» (подвійний клік → редагування).

> Теорію детально викладено в розділі «Теорія функціоналу густини» книги PySCF Book. Тут — лише найпотрібніше.

## Підготовка середовища

In [ ]:
# У Google Colab розкоментуйте наступний рядок (локально PySCF вже має бути встановлений):
# !pip install pyscf

import time
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import minimize_scalar
from scipy.signal import find_peaks

from pyscf import gto, scf, dft, cc, fci
import pyscf

np.set_printoptions(precision=6, suppress=True)
print("PySCF", pyscf.__version__)

HA2EV   = 27.211386      # Ha -> eV
HA2KCAL = 627.5095       # Ha -> kcal/mol

### Допоміжна функція

Щоб не писати щоразу однакові рядки, оголосимо функцію, яка за молекулою та назвою функціонала створює потрібний об'єкт:
`RKS` (замкнена оболонка, $S=0$) або `UKS` (відкрита оболонка). Для `xc='hf'` повертається звичайний RHF/UHF — він знадобиться як «нульова точка відліку».

In [ ]:
def make_ks(mol, xc, level=4):
    """Повертає об'єкт SCF: HF (xc='hf') або Кона-Шема (RKS/UKS) з функціоналом xc."""
    if xc.lower() == 'hf':
        return scf.RHF(mol) if mol.spin == 0 else scf.UHF(mol)
    mf = dft.RKS(mol) if mol.spin == 0 else dft.UKS(mol)
    mf.xc = xc
    mf.grids.level = level          # густина інтеграційної сітки (0..9)
    return mf

# Швидка перевірка
mol = gto.M(atom='H 0 0 0; H 0 0 0.74', basis='def2-svp', verbose=0)
print(make_ks(mol, 'pbe').kernel())

---
# Теоретична довідка (скорочено)

**Ідея.** Багатоелектронна хвильова функція $\Psi(\mathbf r_1,\dots,\mathbf r_N)$ залежить від $3N$ змінних. Електронна густина
$$\rho(\mathbf r)=N\int|\Psi(\mathbf r,\mathbf r_2,\dots,\mathbf r_N)|^2\,d\mathbf r_2\dots d\mathbf r_N,\qquad \int\rho(\mathbf r)\,d\mathbf r=N$$
залежить лише від **трьох** змінних, незалежно від $N$.

**Теореми Хоенберга–Кона.** (1) Густина основного стану $\rho_0(\mathbf r)$ однозначно (з точністю до сталої) визначає зовнішній потенціал, а отже і все про систему. (2) Існує функціонал $E[\rho]$, що досягає мінімуму на точній $\rho_0$ (варіаційний принцип).

**Метод Кона–Шема.** Замість невідомого функціонала кінетичної енергії вводять допоміжну систему *невзаємодіючих* електронів з тією самою густиною $\rho(\mathbf r)=\sum_i^{occ}|\varphi_i(\mathbf r)|^2$:
$$E[\rho]=\underbrace{T_s[\{\varphi_i\}]}_{\text{кінетична}}+\underbrace{\int v_\text{ext}\rho\,d\mathbf r}_{\text{ядра}}+\underbrace{\tfrac12\iint\frac{\rho(\mathbf r)\rho(\mathbf r')}{|\mathbf r-\mathbf r'|}d\mathbf r\,d\mathbf r'}_{J[\rho]\ \text{(Хартрі)}}+E_{xc}[\rho].$$
Мінімізація дає рівняння Кона–Шема, схожі на рівняння Хартрі–Фока:
$$\Big[-\tfrac12\nabla^2+v_\text{ext}(\mathbf r)+v_J(\mathbf r)+v_{xc}(\mathbf r)\Big]\varphi_i=\varepsilon_i\varphi_i,\qquad v_{xc}=\frac{\delta E_{xc}}{\delta\rho}.$$
Усі «складнощі» багатоелектронної задачі сховано в **обмінно-кореляційному функціоналі** $E_{xc}[\rho]$, точного вигляду якого ми не знаємо. Тому DFT — це *точна теорія з наближеним функціоналом*.

**Порівняння з Хартрі–Фоком.** У HF обмін обчислюється точно, а кореляції немає. У KS-DFT обмін і кореляція описуються разом наближеним $E_{xc}[\rho]$. Гібридні функціонали домішують частку точного HF-обміну.

**Драбина Якова** (Perdew): від простих функціоналів до складніших, дедалі точніших і дорожчих.

| Щабель | Залежить від | Приклади (`mf.xc = ...`) |
|---|---|---|
| 1. LDA | $\rho$ | `'lda,vwn'`, `'lda,pw'` |
| 2. GGA | $\rho,\ \nabla\rho$ | `'pbe'`, `'blyp'`, `'bp86'` |
| 3. meta-GGA | $\rho,\ \nabla\rho,\ \tau$ | `'tpss'`, `'scan'`, `'m06l'` |
| 4. Гібриди | + частка точного HF-обміну | `'b3lyp'`, `'pbe0'`, `'m062x'` |
| 4+. Range-separated | HF-обмін залежить від відстані $r_{12}$ | `'camb3lyp'`, `'wb97x'` |

**Алгоритм у PySCF** (той самий SCF-цикл, що й у HF):
```python
mol = gto.M(atom='...', basis='...', spin=...)
mf  = dft.RKS(mol)      # або dft.UKS(mol)
mf.xc = 'pbe'           # <- єдине, що відрізняє DFT від HF
mf.grids.level = 4      # сітка для інтегрування E_xc
e = mf.kernel()
```

---
# Частина 1. Перший розрахунок Кона–Шема

## 1.0 Приклад: атом Гелію

Почнемо з найпростішої багатоелектронної системи — атома He (2 електрони). Для нього відомо **майже точне** значення нерелятивістської повної енергії: $E_\text{exact}=-2.903724$ Ha.

Запустіть клітинку й прочитайте вивід: з яких доданків складається енергія?

In [ ]:
E_EXACT_HE = -2.903724

mol_he = gto.M(atom='He 0 0 0', basis='aug-cc-pvtz', verbose=0)

mf = dft.RKS(mol_he)
mf.xc = 'lda,vwn'          # LDA: обмін Дірака + кореляція VWN
mf.grids.level = 5
e_lda = mf.kernel()

print(f"E(LDA)   = {e_lda: .6f} Ha")
print(f"E(exact) = {E_EXACT_HE: .6f} Ha")
print(f"Похибка  = {(e_lda-E_EXACT_HE)*1000: .1f} mHa")
print()
print("Складові енергії (mf.scf_summary):")
for k, v in mf.scf_summary.items():
    print(f"  {k:5s} = {v: .6f} Ha")
print(f"Енергія орбіталі HOMO: eps = {mf.mo_energy[0]:.4f} Ha")

Тут `e1` — одноелектронна енергія (кінетична + притягання до ядра), `coul` — енергія Хартрі $J[\rho]$, `exc` — обмінно-кореляційна енергія $E_{xc}[\rho]$. Для атома `nuc` (ядерне відштовхування) дорівнює нулю.

## Завдання 1.1. Скільки змінних? *(на розуміння)*

Скільки просторових змінних має (а) хвильова функція атома Ne та (б) його електронна густина $\rho(\mathbf r)$? Як цей факт пояснює, чому DFT дешевший за хвильові методи для великих систем?

**Ваша відповідь:**

(а) ...

(б) ...

Висновок: ...

## Завдання 1.2. HF проти DFT для He *(повторити за зразком)*

Використовуючи зразок із 1.0, обчисліть енергію атома He в базисі `aug-cc-pvtz` для **чотирьох** методів: `'hf'`, `'lda,vwn'`, `'pbe'`, `'b3lyp'` (функція `make_ks` вже вміє все це). Для кожного виведіть $E$, похибку відносно $E_\text{exact}$ у mHa та $\varepsilon_\text{HOMO}$.

*Підказка:* для He (замкнена оболонка) HOMO — це орбіталь з номером `mol.nelectron//2 - 1`.

In [ ]:
methods = ['hf', 'lda,vwn', 'pbe', 'b3lyp']
results_he = {}

for xc in methods:
    # TODO 1: створіть mf = make_ks(mol_he, xc, level=5) та запустіть mf.kernel()
    # TODO 2: збережіть у results_he[xc] кортеж (E, eps_HOMO)
    pass

# TODO 3: виведіть таблицю: метод | E, Ha | похибка, mHa | eps_HOMO, Ha

**Питання.** (1) Який метод дав енергію, найближчу до точної, а який — найгіршу? (2) Енергія HF завжди *вища* за точну (чому?). Чи завжди так для DFT-енергій? Перевірте за таблицею. (3) Точна енергія іонізації He — $0.9037$ Ha. Для точного функціонала $-\varepsilon_\text{HOMO}$ дорівнювала б їй (теорема Янака). Який метод найближчий до цього?

**Ваша відповідь:**

1. ...
2. ...
3. ...

## Завдання 1.3. Баланс енергії *(аналіз)*

Енергія КS складається з $E = T_s + E_\text{ne} + J + E_{xc}$. У `scf_summary` кінетична й ядерно-електронна енергії об'єднані в `e1`. Перевірте для PBE:

1. що `e1 + coul + exc` збігається з `mf.e_tot` (для атома);
2. що `e1` справді дорівнює $\operatorname{Tr}[\mathbf D\,(\mathbf T+\mathbf V_\text{ne})]$, де $\mathbf D$ — матриця густини (`mf.make_rdm1()`), а $\mathbf T$, $\mathbf V_\text{ne}$ — інтеграли `int1e_kin`, `int1e_nuc` (`mol.intor(...)`);
3. окремо виведіть $T_s=\operatorname{Tr}[\mathbf D\mathbf T]$ та $E_\text{ne}=\operatorname{Tr}[\mathbf D\mathbf V_\text{ne}]$.

In [ ]:
mf = make_ks(mol_he, 'pbe', level=5)
mf.kernel()
s = mf.scf_summary

# TODO 1: порівняйте s['e1'] + s['coul'] + s['exc'] з mf.e_tot
# TODO 2: dm = mf.make_rdm1(); T = mol_he.intor('int1e_kin'); V = mol_he.intor('int1e_nuc')
#         обчисліть np.einsum('ij,ji', dm, T+V) і порівняйте з s['e1']
# TODO 3: окремо виведіть T_s = Tr[D T] та Tr[D V]

Зверніть увагу: $T_s>0$ — кінетична енергія *невзаємодіючих* електронів, а не точна кінетична енергія системи. Різницю $T-T_s$ теж «ховають» усередині $E_{xc}$.

---
# Частина 2. Електронна густина на сітці

## Теорія

В DFT електронна густина обчислюється з орбіталей, а енергія $E_{xc}[\rho]$ — **чисельним інтегруванням** на сітці атомно-центрованих точок $\{\mathbf r_g, w_g\}$:
$$N=\int\rho(\mathbf r)\,d\mathbf r\approx\sum_g w_g\,\rho(\mathbf r_g),\qquad E_{xc}\approx\sum_g w_g\,f\big(\rho(\mathbf r_g),\nabla\rho(\mathbf r_g),\dots\big).$$
Якість сітки задається `mf.grids.level` (0 — груба, 9 — дуже щільна; за замовчуванням 3). Перевірка $\int\rho\approx N$ — простий тест на те, що сітка достатня.

## 2.0 Приклад: густина He, радіальний розподіл

Густину обчислюємо з матриці густини та значень АО в точках: $\rho(\mathbf r)=\sum_{\mu\nu}D_{\mu\nu}\chi_\mu(\mathbf r)\chi_\nu(\mathbf r)$. Для атома зручно дивитись на **радіальний розподіл** $4\pi r^2\rho(r)$ — ймовірність знайти електрон у тонкому сферичному шарі радіуса $r$.

In [ ]:
def rho_at_points(mf, coords):
    """Електронна густина rho(r) у довільних точках coords (N,3), у бор^-3."""
    mol = mf.mol
    ao = mf._numint.eval_ao(mol, coords, deriv=0)          # значення АО в точках
    dm = mf.make_rdm1()
    if dm.ndim == 3:                                        # UKS: alpha + beta
        dm = dm[0] + dm[1]
    return mf._numint.eval_rho(mol, ao, dm, xctype='LDA')

mf = make_ks(mol_he, 'pbe', level=5); mf.kernel()

# (1) перевірка нормування на сітці PySCF
rho_grid = rho_at_points(mf, mf.grids.coords)
print("N = ∫rho dr =", np.dot(mf.grids.weights, rho_grid), " (має бути 2)")

# (2) радіальний розподіл вздовж осі z
r = np.linspace(1e-3, 5, 1000)                              # у бор
pts = np.zeros((r.size, 3)); pts[:, 2] = r
rho_r = rho_at_points(mf, pts)

plt.plot(r, 4*np.pi*r**2*rho_r)
plt.xlabel('r, бор'); plt.ylabel(r'$4\pi r^2 \rho(r)$'); plt.title('He, PBE: радіальний розподіл густини')
plt.show()
print("Максимум при r =", r[np.argmax(4*np.pi*r**2*rho_r)], "бор  (радіус Бора a0 = 1)")

## Завдання 2.1. Збіжність за сіткою *(порівняння)*

Для атома Ne (базис `def2-tzvp`, функціонал PBE) виконайте розрахунки з `grids.level` від 0 до 6. Для кожного рівня виведіть:
кількість точок сітки, $\int\rho\,d\mathbf r$ та його відхилення від 10, повну енергію й час розрахунку.
Збудуйте графік $|\Delta E|$ (відносно `level=9`) у логарифмічному масштабі проти level.

*Підказка:* кількість точок — `len(mf.grids.weights)`; час — `time.time()`.

In [ ]:
mol_ne = gto.M(atom='Ne 0 0 0', basis='def2-tzvp', verbose=0)

levels = [0, 1, 2, 3, 4, 5, 6]
data = []
for lev in levels:
    # TODO 1: mf = make_ks(mol_ne, 'pbe', level=lev); виміряйте час kernel()
    # TODO 2: обчисліть N = sum(w*rho) через rho_at_points(mf, mf.grids.coords)
    # TODO 3: додайте у data кортеж (lev, npoints, N, E, t)
    pass

# TODO 4: розрахунок-еталон при level=9 і таблиця: lev | npts | N-10 | E-E_ref (mHa) | t, c
# TODO 5: графік log10|E-E_ref| від level

**Питання.** Починаючи з якого `level` похибка стала менше $10^{-5}$ Ha (≈ 0.01 mHa)? Чому для розрахунків *відносних* енергій (різниць) цього зазвичай достатньо, а для абсолютних — ні? Чи є сенс у `level=9` для рутинної роботи?

**Ваша відповідь:** ...

## Завдання 2.2. Оболонкова структура атомів *(інтерпретація)*

Радіальний розподіл $4\pi r^2\rho(r)$ показує **оболонкову структуру** атома: кількість максимумів дорівнює числу заповнених оболонок.

1. Обчисліть радіальні розподіли для **Ne** та **Ar** (PBE, `def2-tzvp`) вздовж осі $z$ на інтервалі $r\in[0.001, 6]$ бор.
2. Знайдіть положення максимумів за допомогою `find_peaks` і виведіть їх.
3. Побудуйте обидва графіки на одному малюнку.
4. Скільки максимумів у кожного атома? Як це зв'язано з електронною конфігурацією ($1s^22s^22p^6$ та $1s^22s^22p^63s^23p^6$)?

*Підказка:* для `find_peaks` задайте `prominence=0.05`, щоб відфільтрувати дрібні нерегулярності.

In [ ]:
r = np.linspace(1e-3, 6, 3000)
pts = np.zeros((r.size, 3)); pts[:, 2] = r

radial = {}
for atom in ['Ne', 'Ar']:
    # TODO 1: mol = gto.M(atom=atom, basis='def2-tzvp', verbose=0); mf = make_ks(mol, 'pbe', 5); mf.kernel()
    # TODO 2: radial[atom] = 4*pi*r^2*rho(r) (через rho_at_points)
    pass

# TODO 3: знайдіть піки за допомогою find_peaks(..., prominence=0.05) і виведіть r піків
# TODO 4: побудуйте обидва графіки на одному рисунку

**Ваша відповідь:** ...

---
# Частина 3. Від Томаса–Фермі до Кона–Шема

## Теорія

У моделі Томаса–Фермі (ТФ) кінетична енергія — **явний функціонал густини** (локальне наближення з електронного газу):
$$T_\text{TF}[\rho]=C_F\int\rho^{5/3}(\mathbf r)\,d\mathbf r,\qquad C_F=\tfrac{3}{10}(3\pi^2)^{2/3}\approx 2.871.$$
Кон і Шем відмовились від такого наближення: $T_s$ береться з орбіталей, тобто **точно** для невзаємодіючої системи. Це дорожче, але набагато точніше й дає оболонкову структуру, якої ТФ не має.

## 3.0 Приклад: $T_\text{TF}$ проти $T_s$ для He

In [ ]:
C_F = 0.3 * (3*np.pi**2)**(2/3)

def kinetic_TF_and_KS(mf):
    """Повертає (T_TF[rho], T_s) для сконвергованого об'єкта mf."""
    coords, w = mf.grids.coords, mf.grids.weights
    rho = rho_at_points(mf, coords)
    t_tf = C_F * np.dot(w, rho**(5/3))
    dm = mf.make_rdm1()
    if dm.ndim == 3: dm = dm[0] + dm[1]
    t_ks = np.einsum('ij,ji', mf.mol.intor('int1e_kin'), dm)
    return t_tf, t_ks

mf = make_ks(mol_he, 'pbe', level=5); mf.kernel()
t_tf, t_ks = kinetic_TF_and_KS(mf)
print(f"He:  T_TF = {t_tf:.4f} Ha,  T_s(KS) = {t_ks:.4f} Ha,  T_TF/T_s = {t_tf/t_ks:.3f}")

## Завдання 3.1. Наскільки добра модель Томаса–Фермі? *(узагальнення)*

Для **He, Ne, Ar** (базис `def2-tzvp`, PBE, `level=5`) обчисліть $T_\text{TF}$, $T_s$, відношення $T_\text{TF}/T_s$ і віріальну перевірку $-E/T_s$ (для точної теорії $-E/T=1$).
Побудуйте графік відношення $T_\text{TF}/T_s$ залежно від $Z$.

Яка тенденція? Чи узгоджується вона з відомим твердженням, що модель ТФ стає точнішою для **важких** атомів?

In [ ]:
atoms = {'He': 2, 'Ne': 10, 'Ar': 18}
ratio = {}

for sym, Z in atoms.items():
    # TODO 1: побудуйте mol (def2-tzvp) та mf (PBE, level 5), запустіть kernel()
    # TODO 2: t_tf, t_ks = kinetic_TF_and_KS(mf);  ratio[sym] = t_tf / t_ks
    # TODO 3: виведіть Z, T_TF, T_s, T_TF/T_s, -E/T_s
    pass

# TODO 4: графік ratio від Z

**Ваша відповідь:** ...

---
# Частина 4. Драбина Якова на атомах

## Теорія

Чим вище щабель драбини, тим більше інформації про густину використовує функціонал, і тим точніший (як правило) результат. Для оцінки якості функціонала порівнюють з **еталоном** — точними значеннями, які можна отримати високоточними хвильовими методами (CCSD(T), FCI) або з експерименту.

* **LDA**: $E_{xc}^\text{LDA}=\int\rho\,\varepsilon_{xc}^\text{hom}(\rho)\,d\mathbf r$; обмін відомий точно: $\varepsilon_x=-\tfrac34\left(\tfrac3\pi\right)^{1/3}\rho^{1/3}$.
* **GGA** додає залежність від $|\nabla\rho|$ — це дає краще описання неоднорідної густини в молекулах.
* **meta-GGA** додає густину кінетичної енергії $\tau=\tfrac12\sum_i|\nabla\varphi_i|^2$.
* **Гібриди**: $E_{xc}=aE_x^\text{HF}+(1-a)E_x^\text{DFA}+E_c^\text{DFA}$ (наприклад, PBE0: $a=0.25$).

## 4.0 Приклад: He, п'ять щаблів

In [ ]:
mol_he = gto.M(atom='He 0 0 0', basis='aug-cc-pvqz', verbose=0)
print(f"{'xc':10s} {'щабель':14s} {'E, Ha':>11s} {'похибка, mHa':>13s}")
for xc, lvl in [('lda,vwn','1 LDA'), ('pbe','2 GGA'), ('tpss','3 meta-GGA'), ('b3lyp','4 гібрид'), ('pbe0','4 гібрид')]:
    e = make_ks(mol_he, xc, level=5).kernel()
    print(f"{xc:10s} {lvl:14s} {e:11.6f} {(e-E_EXACT_HE)*1000:13.1f}")

## Завдання 4.1. Атом Ne: систематичне порівняння *(порівняння)*

Точна нерелятивістська енергія Ne: $E_\text{exact}=-128.9376$ Ha. Для атома Ne у базисі `aug-cc-pvtz` обчисліть повну енергію функціоналами

`['hf', 'lda,vwn', 'pbe', 'blyp', 'tpss', 'scan', 'b3lyp', 'pbe0', 'camb3lyp']`

і збудуйте стовпчикову діаграму похибок $E-E_\text{exact}$ в mHa.
Обчисліть також CCSD(T)-енергію (еталон хвильового методу); для цього використайте `cc.CCSD(rhf).run()` та `.ccsd_t()`, не забувши додати $E_{(T)}$ до `ccsd.e_tot`.

In [ ]:
E_EXACT_NE = -128.9376
mol_ne = gto.M(atom='Ne 0 0 0', basis='aug-cc-pvtz', verbose=0)

xcs = ['hf', 'lda,vwn', 'pbe', 'blyp', 'tpss', 'scan', 'b3lyp', 'pbe0', 'camb3lyp']
err_ne = {}

for xc in xcs:
    # TODO 1: обчисліть енергію й запишіть похибку (в mHa) у err_ne[xc]
    pass

# TODO 2: CCSD(T) для Ne: rhf = scf.RHF(mol_ne).run(); ccsd = cc.CCSD(rhf).run(); e_t = ccsd.ccsd_t()
#         e_ccsdt = ccsd.e_tot + e_t;  err_ne['CCSD(T)'] = ...
# TODO 3: виведіть таблицю й побудуйте діаграму plt.bar(...) (підпишіть осі)

**Питання.** (1) У якому порядку розташовано функціонали за модулем похибки? Чи відповідає він порядку драбини Якова? (2) Чому похибки повної енергії у DFT у десятки–сотні разів більші за «хімічну точність» (1 ккал/моль ≈ 1.6 mHa), але DFT все одно успішно застосовують в хімії? (3) Чому як еталон варто брати саме CCSD(T)?

**Ваша відповідь:**

1. ...
2. ...
3. ...

---
# Частина 5. Похибка самодії (self-interaction error)

## Теорія

В атомі водню один електрон, тому **електрон не взаємодіє сам із собою**. У точній теорії для довільної одноелектронної густини
$$J[\rho]+E_x[\rho]=0,\qquad E_c[\rho]=0,$$
тобто енергія Хартрі (хибне відштовхування електрона від себе самого) повністю компенсується обміном. У HF це виконується *автоматично* ($J_{11}=K_{11}$). Наближені функціонали (LDA, GGA) компенсують лише частково, і залишається хибна **самодія**:
$$\text{SIE}=J[\rho]+E_{xc}^\text{DFA}[\rho]\neq0.$$
Це одна з головних причин, чому LDA/GGA дають занадто «розмиті» (делокалізовані) електронні густини, занадто малі бар'єри реакцій та занадто мілкі HOMO.

Точна енергія атома H: $E=-0.5$ Ha (нерелятивістська).

## 5.0 Приклад: HF для атома H
У HF-розрахунку обмінна енергія — це $E_x=-\tfrac12\operatorname{Tr}[\mathbf D\mathbf K]$. Перевіримо, що $J+E_x=0$.

In [ ]:
mol_h = gto.M(atom='H 0 0 0', basis='aug-cc-pvqz', spin=1, verbose=0)

mf_hf = scf.UHF(mol_h); e_hf = mf_hf.kernel()
dm = mf_hf.make_rdm1()                      # dm[0] - alpha, dm[1] - beta (нульова)
dm_tot = dm[0] + dm[1]

J_hf  = 0.5 * np.einsum('ij,ji', dm_tot, mf_hf.get_j(mol_h, dm_tot))      # енергія Хартрі
Ex_hf = -0.5 * (np.einsum('ij,ji', dm[0], mf_hf.get_k(mol_h, dm[0]))
              + np.einsum('ij,ji', dm[1], mf_hf.get_k(mol_h, dm[1])))      # точний обмін (HF)

print(f"E(UHF) = {e_hf:.6f}   (точне -0.500000)")
print(f"J = {J_hf:.6f},  Ex = {Ex_hf:.6f},  J+Ex = {J_hf+Ex_hf:.2e}  <- самодія відсутня")

## Завдання 5.1. Самодія в LDA, PBE, B3LYP *(застосування)*

Для атома H (той самий базис, `UKS`, `level=5`) для кожного з функціоналів `['lda,vwn', 'pbe', 'b3lyp']` обчисліть:

* повну енергію $E$ і її відхилення від $-0.5$ Ha (у mHa);
* енергію Хартрі $J$ (формула вище; `dm_tot = dm[0]+dm[1]`);
* $E_{xc}$ з `mf.scf_summary['exc']`;
* самодію $\text{SIE}=J+E_{xc}$ у mHa.

*Підказка:* для гібридного функціонала `scf_summary['exc']` вже містить частку точного HF-обміну. Для точної теорії $E_c=0$ для H, а в наближеннях — ні; нас цікавить саме сума $J+E_{xc}$.

In [ ]:
rows = []
for xc in ['lda,vwn', 'pbe', 'b3lyp']:
    # TODO 1: mf = make_ks(mol_h, xc, level=5); e = mf.kernel()
    # TODO 2: dm_tot = сума alpha і beta матриць; J = 0.5*Tr[D_tot J[D_tot]], де J = mf.get_j(mol_h, dm_tot)
    # TODO 3: exc = mf.scf_summary['exc']; sie = J + exc
    # TODO 4: rows.append((xc, e, J, exc, sie))
    pass

# TODO 5: виведіть таблицю: xc | E, Ha | E+0.5, mHa | J | Exc | SIE, mHa

**Питання.** (1) Який функціонал має найбільшу самодію? (2) Для атома H самодія в PBE дуже мала, а $E\approx-0.5$ Ha. Чи означає це, що PBE «вільний від самодії»? Перевірте себе за результатами Частини 6 (H$_2^+$). (3) У B3LYP самодія більша за PBE, хоч B3LYP містить 20% точного обміну. Чому одноелектронний тест не зводиться до простої залежності від частки HF? (підказка: $E_c$ для одного електрона має бути нулем.)

**Ваша відповідь:**

1. ...
2. ...
3. ...

---
# Частина 6. Де DFT помиляється: розрив зв'язку та делокалізація

## Теорія

**Статична кореляція.** При розтягуванні зв'язку H–H правильна хвильова функція — суперпозиція двох детермінантів ($\sigma_g^2$ та $\sigma_u^2$). Один обмежений детермінант (RHF/RKS) цього не вміє: у ньому 50% внеску дає йонна структура H$^+$H$^-$, тому $E(R\to\infty)\neq2E(\text{H})$.

**«Лікування» порушенням симетрії.** Дозволяємо $\alpha$- та $\beta$-електронам мати різні просторові орбіталі (UHF/UKS). Енергія стає правильною, але ціною *спінового забруднення*: $\langle S^2\rangle\approx1$ замість 0 (суміш синглету й триплету).

**Помилка делокалізації.** Через самодію наближені функціонали «прагнуть» розмазати заряд. Класична ілюстрація — йон H$_2^+$ (1 електрон!) при розтягуванні: точна енергія $E(R\to\infty)=-0.5$ Ha (атом H + протон), але LDA/GGA дають значно **нижчі** значення, бо віддають перевагу «половині електрона на кожному протоні».

## 6.0 Приклад: H$_2$ при $R=3$ Å
Обчислимо енергію методами: RKS (PBE), UKS (PBE, порушена симетрія), FCI (точний). Щоб змусити UKS піти в «правильне» розв'язання, дамо йому стартову матрицю густини, де $\alpha$- і $\beta$-орбіталі — суперпозиції $\sigma_g$ та $\sigma_u$ з різними знаками.

In [ ]:
def h2_energies(R, xc='pbe', basis='cc-pvtz', mix=0.3):
    """Повертає E(RKS), E(UKS з порушеною симетрією), <S^2>(UKS), E(FCI) для H2 при відстані R (Å)."""
    mol = gto.M(atom=f'H 0 0 0; H 0 0 {R}', basis=basis, verbose=0)

    rks = make_ks(mol, xc); e_rks = rks.kernel()

    c = rks.mo_coeff                                    # c[:,0] = sigma_g, c[:,1] = sigma_u
    ca = c[:, [0]]*np.cos(mix) + c[:, [1]]*np.sin(mix)  # alpha: локалізація на лівому атомі
    cb = c[:, [0]]*np.cos(mix) - c[:, [1]]*np.sin(mix)  # beta : на правому
    dm0 = np.array([ca @ ca.T, cb @ cb.T])

    uks = dft.UKS(mol); uks.xc = xc; uks.grids.level = 4
    e_uks = uks.kernel(dm0=dm0)
    s2 = uks.spin_square()[0]

    rhf = scf.RHF(mol).run()
    e_fci = fci.FCI(rhf).kernel()[0]
    return e_rks, e_uks, s2, e_fci

R = 3.0
e_rks, e_uks, s2, e_fci = h2_energies(R)
print(f"R = {R} Å:   RKS = {e_rks:.4f}   UKS(BS) = {e_uks:.4f}  <S^2> = {s2:.3f}   FCI = {e_fci:.4f}")
print(f"2 E(H) (точно) = -1.0000")

## Завдання 6.1. Крива дисоціації H$_2$ *(побудова й інтерпретація)*

1. Використайте `h2_energies` для $R\in\{0.5, 0.74, 1.0, 1.5, 2.0, 2.5, 3.0, 4.0, 5.0\}$ Å (функціонал PBE).
2. Побудуйте **три криві** $E(R)$ (RKS, UKS-BS, FCI) на одному графіку; по осі $y$ — енергія в Ha, обмежте діапазон $[-1.2,-0.8]$.
3. Побудуйте окремий графік $\langle S^2\rangle(R)$ для UKS.
4. Виведіть значення $E(R=5\,\text{Å})-2E(\text{H})$ для кожного методу, де $E(\text{H})=-0.5$ Ha.

In [ ]:
Rs = [0.5, 0.74, 1.0, 1.5, 2.0, 2.5, 3.0, 4.0, 5.0]
E_rks, E_uks, S2, E_fci = [], [], [], []

for R in Rs:
    # TODO 1: викличте h2_energies(R) та додайте результати до списків
    pass

# TODO 2: графік E(R) для трьох методів
# TODO 3: графік <S^2>(R)
# TODO 4: виведіть E(5 Å) + 1.0 для кожного методу (помилка границі дисоціації, mHa)

**Питання.** (1) Скільки становить помилка границі дисоціації в RKS? (2) При яких $R$ UKS починає відрізнятись від RKS і що відбувається з $\langle S^2\rangle$? Чому для малих $R$ розв'язки збігаються? (3) Чи «правильний» UKS-стан? Що він описує — чистий синглет чи суміш? (4) Чому це складність *інтерпретації*, а не лише обчислення?

**Ваша відповідь:**

1. ...
2. ...
3. ...
4. ...

## Завдання 6.2. Делокалізація: йон H$_2^+$ *(перевірка гіпотези)*

Очікування: точна енергія H$_2^+$ при $R\to\infty$ дорівнює $-0.5$ Ha. Перевірте, як справляються HF, PBE та B3LYP:

1. Для $R\in\{1, 2, 3, 4, 5\}$ Å (базис `aug-cc-pvtz`, `charge=1`, `spin=1`) обчисліть $E(R)$ трьома методами.
2. Побудуйте $E(R)$; проведіть горизонтальну лінію $-0.5$.
3. Виведіть похибку при $R=5$ Å: $E(5)+0.5$ (mHa).

*Увага:* на дуже великих $R$ (≳ 8 Å) SCF може збігатися до «локалізованого» розв'язку; тому обмежтесь вказаним інтервалом.

In [ ]:
Rs = [1.0, 2.0, 3.0, 4.0, 5.0]
curves = {}

for xc in ['hf', 'pbe', 'b3lyp']:
    es = []
    for R in Rs:
        # TODO 1: mol = gto.M(atom=f'H 0 0 0; H 0 0 {R}', basis='aug-cc-pvtz', charge=1, spin=1, verbose=0)
        # TODO 2: e = make_ks(mol, xc, level=5).kernel(); es.append(e)
        pass
    curves[xc] = es

# TODO 3: графік E(R) для трьох методів + горизонтальна лінія -0.5
# TODO 4: таблиця E(5 Å) + 0.5 у mHa

**Питання.** (1) Як веде себе HF, і чому (скільки електронів — що з самодією)? (2) Яка з двох функціоналів помиляється сильніше — PBE чи B3LYP, і чому? (3) Що таке «помилка делокалізації» й чим вона небезпечна, наприклад, у задачах переносу заряду?

**Ваша відповідь:**

1. ...
2. ...
3. ...

---
# Частина 7. Орбітальні енергії та потенціал іонізації

## Теорія

Для **точного** функціонала виконується **теорема Янака** (Janak, 1978): енергія HOMO дорівнює мінус потенціалу іонізації,
$$\varepsilon_\text{HOMO}=-\text{IP}.$$
Для наближених функціоналів це порушується, бо їх $v_{xc}(\mathbf r)$ спадає на великих відстанях експоненційно, а не як $-1/r$. Електрон «відчуває» занадто слабке тяжіння й HOMO виходить надто мілкою.

Другий спосіб оцінити IP — **ΔSCF**: $\text{IP}=E(\text{катіон})-E(\text{нейтральна система})$. Він використовує повні енергії й зазвичай точніший.

Експериментальний IP атома Ne: $0.7925$ Ha ($21.56$ еВ).

## 7.0 Приклад: Ne, PBE

In [ ]:
IP_EXP_NE = 0.7925

def ip_delta_scf(sym, xc, basis='aug-cc-pvtz', spin_neutral=0, spin_cation=1, level=5):
    """Повертає (IP за ΔSCF, -eps_HOMO) для атома sym."""
    m0 = gto.M(atom=sym, basis=basis, spin=spin_neutral, verbose=0)
    mf0 = make_ks(m0, xc, level); e0 = mf0.kernel()
    # HOMO нейтральної частинки (для UKS/UHF беремо alpha-орбіталі)
    eps = mf0.mo_energy if mf0.mo_energy.ndim == 1 else mf0.mo_energy[0]
    nocc = m0.nelectron // 2 if m0.spin == 0 else (m0.nelectron + m0.spin)//2
    eps_homo = eps[nocc - 1]
    m1 = gto.M(atom=sym, basis=basis, charge=1, spin=spin_cation, verbose=0)
    e1 = make_ks(m1, xc, level).kernel()
    return e1 - e0, -eps_homo

ip, minus_eps = ip_delta_scf('Ne', 'pbe')
print(f"Ne, PBE:  IP(ΔSCF) = {ip:.4f} Ha,   -eps_HOMO = {minus_eps:.4f} Ha,   експеримент = {IP_EXP_NE}")

## Завдання 7.1. Дві оцінки IP *(порівняння)*

Для Ne (`aug-cc-pvtz`) обчисліть за допомогою `ip_delta_scf` значення IP (ΔSCF) та $-\varepsilon_\text{HOMO}$ для функціоналів
`['hf', 'lda,vwn', 'pbe', 'b3lyp', 'camb3lyp']` і виведіть таблицю з похибками відносно експерименту (в еВ). Побудуйте діаграму: для кожного функціонала поряд два стовпчики — похибка ΔSCF та похибка $-\varepsilon_\text{HOMO}$.

In [ ]:
xcs = ['hf', 'lda,vwn', 'pbe', 'b3lyp', 'camb3lyp']
res_ip = {}

for xc in xcs:
    # TODO 1: res_ip[xc] = ip_delta_scf('Ne', xc)
    pass

# TODO 2: таблиця: xc | IP(ΔSCF), еВ | похибка, еВ | -eps_HOMO, еВ | похибка, еВ
# TODO 3: згрупована діаграма похибок

**Питання.** (1) Яка з двох оцінок точніша для DFT і чому? (2) Для HF помилка у $-\varepsilon_\text{HOMO}$ має протилежний знак порівняно з DFT — чому (теорема Купманса)? (3) Як змінюється $-\varepsilon_\text{HOMO}$ зі збільшенням частки HF-обміну?

**Ваша відповідь:**

1. ...
2. ...
3. ...

## Завдання 7.2. Скільки HF-обміну треба? *(дослідження)*

Власні гібридні функціонали задаються рядком. Наприклад, PBE0 — це `'0.25*HF + 0.75*PBE, PBE'` (обмін: 25% HF + 75% PBE; кореляція: PBE). Загальний вигляд — `'a*HF + (1-a)*PBE, PBE'`.

Для Ne дослідіть залежність від частки $a$ точного обміну: $a\in\{0, 0.25, 0.5, 0.75, 1.0\}$.
Обчисліть IP(ΔSCF) і $-\varepsilon_\text{HOMO}$. Побудуйте обидві залежності від $a$ на одному графіку, додайте горизонтальну лінію експериментального IP.

Знайдіть (приблизно, лінійною інтерполяцією) значення $a$, при якому $-\varepsilon_\text{HOMO}$ збігається з експериментальним IP. Чи збігається воно з $a=0.25$ (PBE0)?

In [ ]:
a_list = [0.0, 0.25, 0.5, 0.75, 1.0]
ip_a, eps_a = [], []

for a in a_list:
    xc = f'{a}*HF + {1-a}*PBE, PBE'
    # TODO: d, ep = ip_delta_scf('Ne', xc); додайте у ip_a, eps_a
    pass

# TODO: графік ip_a(a), eps_a(a) + горизонтальна лінія IP_EXP_NE; оцініть a* лінійною інтерполяцією (np.interp)

**Ваша відповідь:** ...

---
# Частина 8. Молекули: геометрія та енергія атомізації

## Теорія

Для двоатомної молекули AB:

* **Рівноважна довжина зв'язку** $R_e$ — мінімум кривої потенціальної енергії $E(R)$. Знаходимо його мінімізацією функції однієї змінної.
* **Енергія атомізації** (без нульових коливань) $D_e=E(A)+E(B)-E(AB)\big|_{R_e}$. Атоми з відкритою оболонкою рахуємо UKS з відповідним спіном.

Експериментальні значення: N$_2$: $R_e=1.098$ Å, $D_e=228.5$ ккал/моль.

## 8.0 Приклад: N$_2$ з B3LYP

In [ ]:
def energy(atom_str, spin, xc, basis='def2-tzvp', level=4, charge=0):
    mol = gto.M(atom=atom_str, spin=spin, charge=charge, basis=basis, verbose=0)
    return make_ks(mol, xc, level).kernel()

def optimize_diatomic(sym1, sym2, spin, xc, basis='def2-tzvp', bracket=(0.9, 1.1, 1.3), level=4):
    """Мінімізація E(R) за R (Å). Повертає (R_e, E_min)."""
    f = lambda R: energy(f'{sym1} 0 0 0; {sym2} 0 0 {R}', spin, xc, basis, level)
    res = minimize_scalar(f, bracket=bracket, tol=1e-4)
    return res.x, res.fun

t0 = time.time()
Re, Emin = optimize_diatomic('N', 'N', 0, 'b3lyp')
EN = energy('N 0 0 0', 3, 'b3lyp')                    # атом N: квартет (3 неспарені електрони)
De = (2*EN - Emin) * HA2KCAL
print(f"N2 / B3LYP / def2-tzvp:  Re = {Re:.4f} Å (експ. 1.098),   De = {De:.1f} ккал/моль (експ. 228.5)   [{time.time()-t0:.0f} c]")

## Завдання 8.1. Функціонали на N$_2$ *(порівняння)*

За зразком 8.0 для функціоналів `['hf', 'lda,vwn', 'pbe', 'b3lyp']` обчисліть $R_e$ та $D_e$ молекули N$_2$. Виведіть таблицю з похибками відносно експерименту ($R_e=1.098$ Å, $D_e=228.5$ ккал/моль).

**Питання.** (1) Який функціонал найкраще відтворює $R_e$, а який — $D_e$? (2) Який із них *переоцінює*, а який *недооцінює* зв'язок? (3) Чому HF настільки сильно недооцінює $D_e$?

In [ ]:
R_EXP, DE_EXP = 1.098, 228.5
res_n2 = {}

for xc in ['hf', 'lda,vwn', 'pbe', 'b3lyp']:
    # TODO 1: Re, Emin = optimize_diatomic('N', 'N', 0, xc)
    # TODO 2: EN = energy('N 0 0 0', 3, xc); De = (2*EN - Emin)*HA2KCAL
    # TODO 3: res_n2[xc] = (Re, De)
    pass

# TODO 4: таблиця: xc | Re | ΔRe, Å | De | ΔDe, ккал/моль

**Ваша відповідь:**

1. ...
2. ...
3. ...

## Завдання 8.2. Вплив базису *(контроль змінних)*

Змінюючи **лише** базис, дослідіть B3LYP для $D_e$ молекули N$_2$ (при фіксованій геометрії $R=1.098$ Å, щоб не оптимізувати щоразу) в базисах `['sto-3g', '6-31g', 'def2-svp', 'def2-tzvp', 'def2-qzvp']`. Виведіть $D_e$ у ккал/моль і кількість базисних функцій (`mol.nao`); побудуйте $D_e$ проти `nao`.

**Питання.** Який базис можна вважати «досить великим»? Чому базис STO-3G для DFT неприйнятний?

In [ ]:
basis_list = ['sto-3g', '6-31g', 'def2-svp', 'def2-tzvp', 'def2-qzvp']
nao_list, de_list = [], []

for b in basis_list:
    # TODO 1: E(N2) при R=1.098 Å та E(N) (spin=3) у базисі b — функцією energy(...)
    # TODO 2: De = (2*EN - EN2) * HA2KCAL; nao = gto.M(atom='N 0 0 0; N 0 0 1.098', basis=b, verbose=0).nao
    pass

# TODO 3: таблиця й графік De(nao)

**Ваша відповідь:** ...

---
# Частина 9. Технічні параметри: вартість проти якості

## Теорія

Вартість DFT-розрахунку визначається в основному (а) розміром базису, (б) типом функціонала (гібридні дорожчі за чисті GGA, бо потребують обчислення обмінної матриці $K$, як у HF), (в) сіткою. PySCF дозволяє пришвидшити розрахунок прийомом **density fitting**: `mf = mf.density_fit()` — наближене подання чотирицентрових інтегралів через трицентрові (допоміжний базис); помилка зазвичай $\sim10^{-4}$ Ha.

## Завдання 9.1. Час і похибка *(оцінка компромісу)*

Для молекули води (`O 0 0 0; H 0.757 0.587 0; H -0.757 0.587 0`, базис `def2-tzvp`) виконайте розрахунки:

| Метод | Налаштування |
|---|---|
| PBE | звичайний |
| PBE | `density_fit()` |
| B3LYP | звичайний |
| B3LYP | `density_fit()` |

Для кожного виміряйте час і енергію, виведіть різницю енергій з density fitting та без нього (в mHa) і прискорення.

**Питання.** (1) Наскільки density fitting змінює енергію? (2) Чи отримали ви прискорення? Чому для такої маленької молекули воно може бути відсутнє? (3) У яких випадках density fitting варто застосовувати? *(Для перевірки спробуйте бензол, `basis='def2-svp'`.)*

In [ ]:
water = 'O 0 0 0; H 0.757 0.587 0; H -0.757 0.587 0'
mol_w = gto.M(atom=water, basis='def2-tzvp', verbose=0)

timings = {}
for xc in ['pbe', 'b3lyp']:
    for df in [False, True]:
        # TODO 1: mf = make_ks(mol_w, xc, level=3); якщо df: mf = mf.density_fit()
        # TODO 2: виміряйте час kernel(), збережіть у timings[(xc, df)] = (E, t)
        pass

# TODO 3: виведіть таблицю й різниці E(DF) - E(без DF) у mHa та прискорення

**Ваша відповідь:** ...

---
# Частина 10. Підсумковий мініпроект: який функціонал обрати?

## Постановка

У реальному дослідженні перед масовими обчисленнями спочатку **тестують** кілька функціоналів на малому наборі систем з відомим експериментом. Ви виконаєте таке тестування для довжин зв'язків.

Експериментальні довжини зв'язку (Å): $\text{H}_2$ — 0.741, HF — 0.917, N$_2$ — 1.098, CO — 1.128.

## Завдання 10.1. Тест функціоналів *(синтез)*

1. Для чотирьох молекул (усі — закриті оболонки) та функціоналів `['hf', 'lda,vwn', 'pbe', 'b3lyp']` (базис `def2-tzvp`, сітка `level=3`) знайдіть $R_e$ за допомогою `optimize_diatomic`. *(Щоб скоротити час, використайте `bracket` близько до експериментальних значень, наприклад `(R0-0.1, R0, R0+0.1)`.)*
2. Складіть таблицю похибок $R_e^\text{calc}-R_e^\text{exp}$ в Å.
3. Для кожного функціонала обчисліть **MAE** (середню абсолютну похибку) і **MSE** (середню знакову похибку).
4. Зробіть висновок: який функціонал ви рекомендували б для цього класу задач і чому? Чи є системна тенденція (завищення/заниження довжини зв'язку) в HF, LDA, PBE?

In [ ]:
exp_Re = {('H','H'): 0.741, ('H','F'): 0.917, ('N','N'): 1.098, ('C','O'): 1.128}
xcs = ['hf', 'lda,vwn', 'pbe', 'b3lyp']
err = {xc: [] for xc in xcs}          # похибки Re по молекулах

for (a, b), R0 in exp_Re.items():
    for xc in xcs:
        # TODO 1: Re, _ = optimize_diatomic(a, b, 0, xc, bracket=(R0-0.1, R0, R0+0.1), level=3)
        # TODO 2: err[xc].append(Re - R0)
        pass

# TODO 3: таблиця (рядки — молекули, стовпці — функціонали) + MAE, MSE для кожного функціонала
# TODO 4: (необов'язково) діаграма MAE

**Ваш висновок:** ...

---
# Контрольні запитання (для самоперевірки)

1. Сформулюйте першу та другу теореми Хоенберга–Кона. Чому друга теорема дає змогу використовувати варіаційний принцип?
2. У чому різниця між $T$ та $T_s$? Куди «ховається» різниця між ними в методі Кона–Шема?
3. Чому метод Кона–Шема називають «точним у принципі, але наближеним на практиці»?
4. Перелічіть щаблі драбини Якова та вкажіть, яку нову інформацію про густину додає кожен наступний щабель.
5. Що таке самодія? Як вона проявляється в енергії атома H, у HOMO та в розтягнутому H$_2^+$?
6. Чому гібридні функціонали зменшують самодію? Чому одна й та сама частка HF-обміну (25%) не оптимальна для всіх властивостей?
7. Поясніть, чому RKS не годиться для розриву зв'язку H–H, а UKS дає правильну енергію, проте «неправильний» $\langle S^2\rangle$.
8. Чому DFT зазвичай точніший для різниць енергій, ніж для повних енергій?
9. Чому базисний набір потрібно обирати «хімічно розумно», а `grids.level` — перевіряти?
10. Коли варто надавати перевагу чистому GGA, а коли гібриду? Як змінюється вартість?

## Для допитливих *(необов'язково)*

* **Дисперсійна взаємодія.** Стандартні LDA/GGA/гібриди не описують дальню ван-дер-Ваальсову взаємодію (Ar···Ar, π-стекінг). Додають емпіричні поправки (Grimme D3/D4, пакет `pyscf-dispersion`: `mf.disp = 'd3bj'`) або використовують нелокальні функціонали (VV10, ωB97X-V).
* **Збуджені стани: TD-DFT** (`from pyscf import tddft; td = tddft.TDDFT(mf)`) — і чому CAM-B3LYP краще за B3LYP для переходів з перенесенням заряду.
* **Нова ідея.** Знайдіть у літературі оптимально налаштовані range-separated гібриди (ωB97X-D, LC-ωPBE з налаштуванням $\omega$) та спробуйте знайти $\omega$, при якому $-\varepsilon_\text{HOMO}=\text{IP}$ (ΔSCF) для Ne.